# Kiểm tra chiến lược dự đoán giá Copper

Notebook tái sử dụng dự đoán đã lưu từ thí nghiệm 14-11 làm đối chứng. Ba khoảng test giữ nguyên để so sánh trên cùng ngày. Notebook chỉ tìm tham số và huấn luyện CNN-LSTM, CNN-GRU; không huấn luyện lại các model cũ và không dùng simulated annealing.

Bài toán ước lượng Copper(t) dùng cửa sổ 22 phiên giá hàng hóa ngoại sinh, bao gồm ngày t. Đây không phải dự báo ngày kế tiếp.


## 1. Nạp dữ liệu và dự đoán đối chứng

Notebook đọc dataset hàng hóa và 21 CSV dự đoán của thí nghiệm 14-11. Trên Kaggle, đính kèm file ZIP dự đoán làm input; khi chạy trong dự án, notebook đọc trực tiếp thư mục kết quả.


In [ ]:
from pathlib import Path
import json, itertools, time
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.stats import spearmanr
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.callbacks import EarlyStopping

DATASET_CANDIDATES = [
    Path(
        "/kaggle/input/datasets/lancelot71000/dataset-9-commodities-daily/dataset_9_commodities_daily.csv"
    ),
    Path("/kaggle/input/dataset-9-commodities-daily/dataset_9_commodities_daily.csv"),
    Path("data/processed/dataset_9_commodities_daily.csv"),
]
DATA_PATH = next((p for p in DATASET_CANDIDATES if p.is_file()), None)
if DATA_PATH is None:
    kaggle_matches = (
        list(Path("/kaggle/input").rglob("dataset_9_commodities_daily.csv"))
        if Path("/kaggle/input").exists()
        else []
    )
    if len(kaggle_matches) != 1:
        raise FileNotFoundError(
            "Không tìm thấy duy nhất dataset_9_commodities_daily.csv."
        )
    DATA_PATH = kaggle_matches[0]

TARGET = "Copper"
CANDIDATE_FEATURES = [
    "Aluminium",
    "Nickel",
    "Tin",
    "Iron_Ore_62",
    "Silver",
    "Gold",
    "WTI",
    "Brent",
]
LOOKBACK = 22
TEST_RATIOS = {"0.9": 0.90, "0.925": 0.925, "0.95": 0.95}
ROLLING_ROWS = 756
SEEDS = [17, 42, 91]
BATCH_SIZE = 64
SEARCH_EPOCHS = 40
FINAL_EPOCHS = 100
PATIENCE = 7
OUTPUT_DIR = Path("/kaggle/working/copper_strategy_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(DATA_PATH)
raw["Date"] = pd.to_datetime(raw["Date"], errors="coerce")
for col in [TARGET] + CANDIDATE_FEATURES:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")
data = (
    raw.dropna(subset=["Date", TARGET] + CANDIDATE_FEATURES)
    .sort_values("Date")
    .drop_duplicates("Date")
    .reset_index(drop=True)
)

# Kaggle extracts the uploaded result bundle under this input folder.
RESULT_ROOT = Path("/kaggle/input/datasets/lancelot71000/result")
if not RESULT_ROOT.is_dir():
    raise FileNotFoundError(f"Missing Kaggle result folder: {RESULT_ROOT}")

OLD_MODELS = [
    "naive_persistence",
    "lstm",
    "paper_sa_lstm",
    "reoptimized_sa_lstm",
    "gru",
    "cnn1d_lstm",
    "sa_gru",
]
expected_names = {
    f"predictions_{model}_{ratio}.csv" for model in OLD_MODELS for ratio in TEST_RATIOS
}
found_paths = {
    path.name: path
    for path in RESULT_ROOT.rglob("predictions_*.csv")
    if path.name in expected_names
}
missing_names = sorted(expected_names - found_paths.keys())
if missing_names:
    raise FileNotFoundError(
        f"Missing {len(missing_names)} prediction CSVs under {RESULT_ROOT}: {missing_names}"
    )
OLD_PREDICTIONS = {
    name: pd.read_csv(found_paths[name], parse_dates=["Date"])
    for name in sorted(expected_names)
}
print(f"Data: {len(data):,} rows; loaded old predictions: {len(OLD_PREDICTIONS)}")


## 2. Chia thời gian, chọn biến và chuẩn hóa

Ba mốc test là 90/10, 92,5/7,5 và 95/5 như thí nghiệm cũ. Ở mỗi mốc, chọn bốn biến bằng Spearman chỉ trên train. Expanding dùng toàn bộ lịch sử; rolling dùng tối đa 756 phiên gần nhất. Scaler fit trên phần train; validation giữ nguyên thứ tự thời gian.


In [ ]:
def choose_features(train_frame):
    correlations = {
        col: abs(spearmanr(train_frame[col], train_frame[TARGET], nan_policy="omit").statistic)
        for col in CANDIDATE_FEATURES
    }
    correlations = {
        col: float(value) if np.isfinite(value) else -1.0
        for col, value in correlations.items()
    }
    selected = sorted(CANDIDATE_FEATURES, key=lambda col: (-correlations[col], col))[:4]
    return selected, correlations


def make_windows(x, y, start, end):
    indices = np.arange(max(start, LOOKBACK - 1), end)
    X = np.asarray([x[i - LOOKBACK + 1:i + 1] for i in indices], dtype="float32")
    Y = np.asarray([y[i] for i in indices], dtype="float32")
    return X, Y, indices


def make_split(ratio_key, ratio, scheme):
    n = len(data)
    test_start = int(n * ratio)
    if scheme == "expanding":
        fit_start = 0
    else:
        fit_start = max(0, test_start - ROLLING_ROWS)
    val_start = fit_start + int((test_start - fit_start) * 0.90)

    train_frame = data.iloc[fit_start:val_start]
    features, correlations = choose_features(train_frame)
    x_scaler = MinMaxScaler().fit(train_frame[features])
    y_scaler = MinMaxScaler().fit(train_frame[[TARGET]])
    x = x_scaler.transform(data[features])
    y = y_scaler.transform(data[[TARGET]]).ravel()

    train = make_windows(x, y, fit_start, val_start)
    val = make_windows(x, y, val_start, test_start)
    test = make_windows(x, y, test_start, n)
    return {
        "ratio_key": ratio_key, "scheme": scheme, "fit_start": fit_start,
        "val_start": val_start, "test_start": test_start, "features": features,
        "correlations": correlations, "x_scaler": x_scaler, "y_scaler": y_scaler,
        "train": train, "val": val, "test": test, "x": x,
    }


## 3. Kiến trúc và tìm tham số CNN

CNN-LSTM và CNN-GRU dùng Conv1D, MaxPooling, hai lớp hồi quy, Dense, Dropout và đầu ra tuyến tính. Lưới tham số được chấm bằng MSE validation đã chuẩn hóa; EarlyStopping dừng ứng viên kém. Test không tham gia chọn tham số. Cấu hình tốt nhất được huấn luyện với ba seed.


In [ ]:
CNN_CONFIG = {
    "filters": [16, 32],
    "kernel_sizes": [2, 3],
    "recurrent_units": [(32, 16), (48, 24)],
    "learning_rates": [0.001, 0.0005],
    "dense_units": 16,
    "dropout": 0.20,
    "batch_size": BATCH_SIZE,
    "max_search_epochs": SEARCH_EPOCHS,
    "final_epochs": FINAL_EPOCHS,
    "early_stopping_patience": PATIENCE,
}


def build_cnn_lstm(input_shape, params):
    model = tf.keras.Sequential(name="CNN_LSTM")
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.Conv1D(
        filters=params["filters"], kernel_size=params["kernel_size"],
        padding="causal", activation="relu",
    ))
    model.add(tf.keras.layers.MaxPooling1D(pool_size=2))
    model.add(tf.keras.layers.LSTM(params["units_1"], return_sequences=True))
    model.add(tf.keras.layers.LSTM(params["units_2"]))
    model.add(tf.keras.layers.Dense(CNN_CONFIG["dense_units"], activation="relu"))
    model.add(tf.keras.layers.Dropout(CNN_CONFIG["dropout"]))
    model.add(tf.keras.layers.Dense(1, activation="linear"))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(params["learning_rate"]),
        loss="mse",
    )
    return model


def build_cnn_gru(input_shape, params):
    model = tf.keras.Sequential(name="CNN_GRU")
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.Conv1D(
        filters=params["filters"], kernel_size=params["kernel_size"],
        padding="causal", activation="relu",
    ))
    model.add(tf.keras.layers.MaxPooling1D(pool_size=2))
    model.add(tf.keras.layers.GRU(params["units_1"], return_sequences=True))
    model.add(tf.keras.layers.GRU(params["units_2"]))
    model.add(tf.keras.layers.Dense(CNN_CONFIG["dense_units"], activation="relu"))
    model.add(tf.keras.layers.Dropout(CNN_CONFIG["dropout"]))
    model.add(tf.keras.layers.Dense(1, activation="linear"))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(params["learning_rate"]),
        loss="mse",
    )
    return model


BUILDERS = {"cnn_lstm": build_cnn_lstm, "cnn_gru": build_cnn_gru}


def parameter_grid():
    return [
        {
            "filters": filters, "kernel_size": kernel,
            "units_1": units[0], "units_2": units[1],
            "learning_rate": learning_rate,
        }
        for filters, kernel, units, learning_rate in itertools.product(
            CNN_CONFIG["filters"], CNN_CONFIG["kernel_sizes"],
            CNN_CONFIG["recurrent_units"], CNN_CONFIG["learning_rates"],
        )
    ]


def select_parameters(model_name, split):
    X_train, y_train, _ = split["train"]
    X_val, y_val, _ = split["val"]
    best = None
    history = []
    for candidate_id, params in enumerate(parameter_grid(), start=1):
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(314 + candidate_id)
        model = BUILDERS[model_name](X_train.shape[1:], params)
        callback = EarlyStopping(
            monitor="val_loss", patience=PATIENCE, restore_best_weights=True
        )
        started = time.perf_counter()
        model.fit(
            X_train, y_train, validation_data=(X_val, y_val),
            epochs=SEARCH_EPOCHS, batch_size=BATCH_SIZE, shuffle=False,
            verbose=0, callbacks=[callback],
        )
        val_prediction = model.predict(X_val, verbose=0).ravel()
        val_mse = float(mean_squared_error(y_val, val_prediction))
        history.append({
            "model": model_name, "candidate": candidate_id, **params,
            "validation_mse_scaled": val_mse,
            "epochs_run": len(model.history.history["loss"]),
            "seconds": time.perf_counter() - started,
        })
        if best is None or val_mse < best["validation_mse_scaled"]:
            best = {"params": params, "validation_mse_scaled": val_mse}
        del model
    return best, history


## 4. Đối chiếu và dịch chuyển chế độ

Dự đoán cũ được đổi ngược về giá gốc rồi đổi sang scaler của từng chế độ train để so sánh trên cùng thang. Notebook thêm Ridge dự đoán phần dư Copper(t)−Copper(t−1). Rolling window và dự đoán phần dư là các cách thử khi mức giá test dịch khỏi vùng train; kết quả được đánh giá theo từng khoảng test.


In [ ]:
def add_prediction(rows, split, model, seed, dates, actual_scaled, predicted_scaled):
    actual_raw = split["y_scaler"].inverse_transform(
        np.asarray(actual_scaled).reshape(-1, 1)
    ).ravel()
    predicted_raw = split["y_scaler"].inverse_transform(
        np.asarray(predicted_scaled).reshape(-1, 1)
    ).ravel()
    for i, date in enumerate(dates):
        rows.append({
            "ratio": split["ratio_key"], "scheme": split["scheme"],
            "date": pd.Timestamp(date), "model": model, "seed": seed,
            "actual_scaled": float(actual_scaled[i]),
            "prediction_scaled": float(predicted_scaled[i]),
            "actual": float(actual_raw[i]), "prediction": float(predicted_raw[i]),
        })


def load_old_predictions(split, old_scaler):
    ratio = split["ratio_key"]
    test_dates = set(data.Date.iloc[split["test_start"]:])
    rows = []
    for model in [
        "naive_persistence", "lstm", "paper_sa_lstm",
        "reoptimized_sa_lstm", "gru", "cnn1d_lstm", "sa_gru",
    ]:
        filename = f"predictions_{model}_{ratio}.csv"
        frame = OLD_PREDICTIONS[filename].copy()
        frame = frame[frame["Date"].isin(test_dates)].copy()
        actual_raw = old_scaler.inverse_transform(frame[["actual_scaled"]]).ravel()
        prediction_raw = old_scaler.inverse_transform(frame[["predicted_scaled"]]).ravel()
        actual_scaled = split["y_scaler"].transform(actual_raw.reshape(-1, 1)).ravel()
        prediction_scaled = split["y_scaler"].transform(prediction_raw.reshape(-1, 1)).ravel()
        add_prediction(
            rows, split, f"14-11_{model}", None, frame["Date"].to_numpy(),
            actual_scaled, prediction_scaled,
        )
    return rows


def metric_table(predictions):
    rows = []
    keys = ["ratio", "scheme", "model", "seed"]
    for key, group in predictions.groupby(keys, dropna=False):
        actual = group["actual_scaled"].to_numpy()
        predicted = group["prediction_scaled"].to_numpy()
        rows.append({
            **dict(zip(keys, key)),
            "MSE_scaled": mean_squared_error(actual, predicted),
            "MAE_scaled": mean_absolute_error(actual, predicted),
            "RMSE_scaled": mean_squared_error(actual, predicted) ** 0.5,
            "R2_scaled": r2_score(actual, predicted) if len(group) > 1 else np.nan,
            "n_test": len(group),
        })
    return pd.DataFrame(rows)


In [ ]:
prediction_rows = []
search_rows = []
feature_rows = []

for ratio_key, ratio in TEST_RATIOS.items():
    test_start = int(len(data) * ratio)
    old_val_start = int(test_start * 0.90)
    old_scaler = MinMaxScaler().fit(data.iloc[:old_val_start][[TARGET]])

    for scheme in ["expanding", "rolling"]:
        split = make_split(ratio_key, ratio, scheme)
        feature_rows.append({
            "ratio": ratio_key, "scheme": scheme,
            "fit_start": split["fit_start"], "val_start": split["val_start"],
            "test_start": split["test_start"],
            "features": ",".join(split["features"]),
            **split["correlations"],
        })

        # Reuse 14-11 model outputs. Convert them into this scheme's train-fitted scale.
        prediction_rows.extend(load_old_predictions(split, old_scaler))

        # Persistence and lag-5 seasonal naive are computed directly on the same test dates.
        X_test, y_test, test_indices = split["test"]
        actual_raw = data[TARGET].iloc[test_indices].to_numpy()
        for model_name, raw_prediction in [
            ("persistence", data[TARGET].iloc[test_indices - 1].to_numpy()),
            ("seasonal_naive_lag5", data[TARGET].iloc[test_indices - 5].to_numpy()),
        ]:
            scaled_prediction = split["y_scaler"].transform(
                raw_prediction.reshape(-1, 1)
            ).ravel()
            add_prediction(
                prediction_rows, split, model_name, None,
                data.Date.iloc[test_indices].to_numpy(), y_test, scaled_prediction,
            )

        # Ridge baseline predicts the one-step Copper change, then adds the prior close.
        train_indices = split["train"][2]
        train_x = np.asarray([
            split["x"][i - LOOKBACK + 1:i + 1].reshape(-1)
            for i in train_indices
        ])
        test_x = np.asarray([
            split["x"][i - LOOKBACK + 1:i + 1].reshape(-1)
            for i in test_indices
        ])
        train_delta = (
            data[TARGET].iloc[train_indices].to_numpy()
            - data[TARGET].iloc[train_indices - 1].to_numpy()
        )
        ridge = Ridge().fit(train_x, train_delta)
        ridge_raw = data[TARGET].iloc[test_indices - 1].to_numpy() + ridge.predict(test_x)
        ridge_scaled = split["y_scaler"].transform(ridge_raw.reshape(-1, 1)).ravel()
        add_prediction(
            prediction_rows, split, "ridge_residual_delta", None,
            data.Date.iloc[test_indices].to_numpy(), y_test, ridge_scaled,
        )

        for model_name in ["cnn_lstm", "cnn_gru"]:
            best, history = select_parameters(model_name, split)
            search_rows.extend([
                {"ratio": ratio_key, "scheme": scheme, **row}
                for row in history
            ])
            X_train, y_train, _ = split["train"]
            X_val, y_val, _ = split["val"]
            X_fit = np.concatenate([X_train, X_val], axis=0)
            y_fit = np.concatenate([y_train, y_val], axis=0)
            for seed in SEEDS:
                tf.keras.backend.clear_session()
                tf.keras.utils.set_random_seed(seed)
                model = BUILDERS[model_name](X_fit.shape[1:], best["params"])
                started = time.perf_counter()
                model.fit(
                    X_fit, y_fit, epochs=FINAL_EPOCHS, batch_size=BATCH_SIZE,
                    shuffle=False, verbose=0,
                )
                prediction = model.predict(X_test, verbose=0).ravel()
                search_rows.append({
                    "ratio": ratio_key, "scheme": scheme, "model": model_name,
                    "record_type": "selected_final",
                    **best["params"],
                    "validation_mse_scaled": best["validation_mse_scaled"],
                    "seed": seed, "final_fit_seconds": time.perf_counter() - started,
                })
                add_prediction(
                    prediction_rows, split, model_name, seed,
                    data.Date.iloc[test_indices].to_numpy(), y_test, prediction,
                )
                del model
        print(f"Completed ratio={ratio_key}, scheme={scheme}")

predictions = pd.DataFrame(prediction_rows)
metrics = metric_table(predictions)
metrics.to_csv(OUTPUT_DIR / "metrics_scaled.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "predictions_comparison.csv", index=False)
pd.DataFrame(search_rows).to_csv(OUTPUT_DIR / "cnn_parameter_search.csv", index=False)
pd.DataFrame(feature_rows).to_csv(OUTPUT_DIR / "feature_selection_by_split.csv", index=False)
archive_path = __import__("shutil").make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
display(metrics.sort_values(["ratio", "scheme", "MSE_scaled"]))
print("Saved:", OUTPUT_DIR, archive_path)


## 5. Chỉ số và tệp xuất

MSE, MAE, RMSE và R² được tính trên thang MinMax của từng tập train. Notebook lưu chỉ số theo mốc test, chế độ train, model và seed; đồng thời lưu dự đoán, tham số đã chọn và ZIP kết quả.
